In [1]:
# 필요 라이브러리 설치 (최초 1회 실행)
# !pip install -q requests pandas plotly ipywidgets openpyxl

import os
import re
import urllib.parse
import requests
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ---------------------------------------------------------
# [1. 영양 DB 로드 및 정밀 수분 추출 로직]
# ---------------------------------------------------------
DB_FOLDER_PATH = r"C:\Users\User\Desktop\밥스누\10. 프로그램 개발\원료 가격 산정 프로그램\원재료 영양DB"
NUTRITION_DB = pd.DataFrame()

def load_nutrition_db():
    """영양 DB 자동 탐색 및 로드"""
    global NUTRITION_DB

    if os.path.exists(DB_FOLDER_PATH):
        files = os.listdir(DB_FOLDER_PATH)
        target_files = [f for f in files if f.endswith(('.xlsx', '.xls', '.csv')) and not f.startswith('~$')]

        for fname in target_files:
            full_path = os.path.join(DB_FOLDER_PATH, fname)
            try:
                if fname.endswith('.csv'):
                    try:
                        NUTRITION_DB = pd.read_csv(full_path, encoding='utf-8')
                    except UnicodeDecodeError:
                        NUTRITION_DB = pd.read_csv(full_path, encoding='cp949')
                else:
                    NUTRITION_DB = pd.read_excel(full_path)
                print(f"✅ [영양 DB 연동 성공] 파일명: {fname}")
                return
            except Exception as e:
                print(f"❌ '{fname}' 읽기 실패: {e}")

    # Fallback 기본 데이터 (DB 미존재 시 사용)
    fallback_data = [
        {'식품명': '현미', '수분(%)': 12.8},
        {'식품명': '케일', '수분(%)': 89.6},
        {'식품명': '브로콜리', '수분(%)': 89.3},
        {'식품명': '비트', '수분(%)': 87.5},
        {'식품명': '양파', '수분(%)': 90.1},
        {'식품명': '당근', '수분(%)': 88.3},
        {'식품명': '배추', '수분(%)': 95.2},
        {'식품명': '무', '수분(%)': 94.4},
        {'식품명': '마늘', '수분(%)': 64.5},
        {'식품명': '파프리카', '수분(%)': 92.2},
        {'식품명': '사과', '수분(%)': 85.7}
    ]
    NUTRITION_DB = pd.DataFrame(fallback_data)
    print("ℹ️ 기본 추정 데이터 세트로 작동합니다.")

load_nutrition_db()

def get_moisture_content_accurate(item_name):
    """
    3단계 정밀 매칭을 통한 수분 함량(%) 도출
    1순위: 완전 일치 ('현미', '케일')
    2순위: 신선 원물 키워드 ('생것', '원물', '생', '신선') 매칭
    3순위: 가장 단어 길이가 짧은 순수 원물 매칭
    """
    if NUTRITION_DB.empty:
        return 90.0

    cols = NUTRITION_DB.columns
    name_col = next((c for c in cols if any(k in str(c) for k in ['식품', '품목', '명', '이름', '원료'])), None)
    water_col = next((c for c in cols if any(k in str(c) for k in ['수분', 'water', 'Moisture'])), None)

    if not name_col or not water_col:
        return 90.0

    clean_target = item_name.replace(" ", "").strip()
    df_temp = NUTRITION_DB.copy()
    df_temp['clean_name'] = df_temp[name_col].astype(str).str.replace(" ", "").str.strip()

    # 1. 완전 일치
    exact = df_temp[df_temp['clean_name'] == clean_target]
    if not exact.empty:
        try:
            return float(str(exact.iloc[0][water_col]).replace(',', ''))
        except ValueError:
            pass

    # 2. 신선 원물 키워드 포함 매칭
    raw_match = df_temp[
        df_temp['clean_name'].str.contains(clean_target, na=False) &
        df_temp['clean_name'].str.contains('생|원물|신선|생것', na=False)
    ]
    if not raw_match.empty:
        try:
            return float(str(raw_match.iloc[0][water_col]).replace(',', ''))
        except ValueError:
            pass

    # 3. 부분 일치 중 최단 문자열 선택
    partial = df_temp[df_temp['clean_name'].str.contains(clean_target, na=False)].copy()
    if not partial.empty:
        partial['name_len'] = partial['clean_name'].str.len()
        partial = partial.sort_values('name_len')
        try:
            return float(str(partial.iloc[0][water_col]).replace(',', ''))
        except ValueError:
            pass

    return 90.0

# ---------------------------------------------------------
# [2. 단위(g, kg, 100g 등) -> 1kg 환산 로직]
# ---------------------------------------------------------
def convert_to_kg_factor(unit_str, unit_sz_val):
    """단위 텍스트와 용량을 분석하여 1kg 환산 계수 산출"""
    u_str = str(unit_str).strip().lower()
    try:
        sz = float(unit_sz_val) if float(unit_sz_val) > 0 else 1.0
    except ValueError:
        sz = 1.0

    # '100g', '500g', '10kg' 등 단위 내 숫자가 포함된 경우
    nums = re.findall(r'[\d\.]+', u_str)
    num_in_unit = float(nums[0]) if nums else 1.0

    if 'g' in u_str and 'kg' not in u_str:
        return (num_in_unit * sz) / 1000.0  # g -> kg
    elif 'kg' in u_str:
        return num_in_unit * sz             # kg -> kg
    else:
        return sz                           # 기타 규격은 sz 기본값 적용

# ---------------------------------------------------------
# [3. API 설정 및 확장된 품목 코드 사전]
# ---------------------------------------------------------
ENDPOINT_URL = "https://apis.data.go.kr/B552845/perYearMonth/price"
RAW_SERVICE_KEY = "JKLUuK7qjHhv4KkExYzgXfzQ7X5bguij14ZcZQcAWmarjw2elFsqtCNQaz3t6WQ4lDjeEfkYAjrv3qm3CVwjIQ%3D%3D"

ITEM_CODE_MULTI_MAP = defaultdict(list)
# 케일, 현미, 찹쌀 등 검색 자주 누락되는 품목 코드 대거 보강
DEFAULT_ITEMS = [
    ('쌀', '111'), ('현미', '112'), ('찹쌀', '113'), ('콩', '141'), ('팥', '142'),
    ('배추', '211'), ('양배추', '212'), ('시금치', '213'), ('상추', '214'),
    ('무', '231'), ('당근', '232'), ('양파', '245'), ('파', '246'), ('생강', '247'),
    ('피마늘', '244'), ('깐마늘(국산)', '258'), ('깐마늘(수입)', '259'), ('파프리카', '256'),
    ('브로콜리', '261'), ('브로콜리', '280'), ('케일', '264'), ('양상추', '262'), ('청경채', '263'),
    ('비트', '265'), ('방울토마토', '422'), ('사과', '411'), ('배', '412'), ('포도', '414'), ('감귤', '415')
]

for name, code in DEFAULT_ITEMS:
    if code not in ITEM_CODE_MULTI_MAP[name]:
        ITEM_CODE_MULTI_MAP[name].append(code)

API_CACHE = {}
df_searched_raw = pd.DataFrame()
df_searched_latest = pd.DataFrame()

# ---------------------------------------------------------
# [4. UI 위젯 구성]
# ---------------------------------------------------------
header_html = widgets.HTML("""
<div style="background-color:#0d47a1; padding:14px; border-radius:8px; color:white;">
    <h2 style="margin:0;">🧪 농산물 원가 산정 System (정밀 수분 DB & 1kg 단가 환산)</h2>
    <p style="margin:4px 0 0 0;">품목 검색 ➔ 1kg 기준 단가 자동 환산 ➔ 수분 분석 ➔ 동결건조 분말 원가 도출</p>
</div>
""")

txt_search = widgets.Text(value="현미", placeholder="품목명 (예: 현미, 케일, 브로콜리)", description="품목명:", layout=widgets.Layout(width="22%"))
dd_se_cd = widgets.Dropdown(
    options=[("전체 유통단계", ""), ("소매", "1"), ("중도매", "2"), ("친환경농산물", "3"), ("친환경농산물(신규)", "7")],
    value="", description="유통구분:", layout=widgets.Layout(width="24%")
)
txt_start_date = widgets.Text(value="202401", description="시작연월:", layout=widgets.Layout(width="18%"))
txt_end_date = widgets.Text(value="202608", description="종료연월:", layout=widgets.Layout(width="18%"))
btn_search = widgets.Button(description="원물 가격 조회", button_style="primary", icon="search")

dd_item_select = widgets.Dropdown(description="원물 선택:", layout=widgets.Layout(width="55%"))
num_raw_weight = widgets.FloatText(value=100.0, description="원물 투입량(kg):", layout=widgets.Layout(width="22%"))
num_moisture = widgets.FloatText(value=12.8, description="수분함량(%):", layout=widgets.Layout(width="22%"))
num_mfg_cost = widgets.FloatText(value=150000.0, description="제조원가(원):", layout=widgets.Layout(width="22%"))
btn_calc_powder = widgets.Button(description="동결건조 원가 계산", button_style="success", icon="calculator")

lbl_status = widgets.HTML(value="<b>상태:</b> 품목명을 입력하고 [원물 가격 조회]를 클릭하세요.")
lbl_powder_result = widgets.HTML(value="<div style='padding:10px; border:1px solid #ddd; border-radius:5px;'>원물 선택 후 원가 계산을 진행하세요.</div>")

out_table = widgets.Output()
out_chart = widgets.Output()

# ---------------------------------------------------------
# [5. API 수집 엔진 및 1kg 환산 로직]
# ---------------------------------------------------------
def fetch_single_page(item_cd, se_code, start_ym, end_ym, page_no=1, num_of_rows=1000):
    request_url = f"{ENDPOINT_URL}?serviceKey={RAW_SERVICE_KEY}"
    params = {
        "returnType": "json",
        "pageNo": str(page_no),
        "numOfRows": str(num_of_rows),
        "cond[exmn_ym::GTE]": start_ym,
        "cond[exmn_ym::LTE]": end_ym,
    }
    if item_cd:
        params["cond[item_cd::EQ]"] = str(item_cd)
    if se_code:
        params["cond[se_cd::EQ]"] = str(se_code)

    try:
        res = requests.get(request_url, params=params, timeout=6, verify=False)
        if res.status_code == 200:
            data = res.json()
            body = data.get("response", {}).get("body", {})
            items_container = body.get("items", {})
            items = items_container.get("item", []) if isinstance(items_container, dict) else []
            if isinstance(items, dict):
                items = [items]
            return items, int(body.get("totalCount", 0))
    except Exception:
        pass
    return [], 0

def fetch_data_smart(item_keyword, se_code, start_ym, end_ym):
    cache_key = f"{item_keyword}_{se_code}_{start_ym}_{end_ym}"
    if cache_key in API_CACHE:
        return API_CACHE[cache_key]

    matched_codes = ITEM_CODE_MULTI_MAP.get(item_keyword, [])
    all_raw_items = []

    if matched_codes:
        with ThreadPoolExecutor(max_workers=5) as executor:
            future_to_code = {
                executor.submit(fetch_single_page, code, se_code, start_ym, end_ym, 1, 1000): code
                for code in matched_codes
            }
            for future in as_completed(future_to_code):
                items, _ = future.result()
                all_raw_items.extend(items)

    # 코드가 없거나 결과가 없으면 전체 목록 스캔 검색 수행
    if not all_raw_items:
        first_page_items, _ = fetch_single_page(None, se_code, start_ym, end_ym, 1, 1000)
        all_raw_items.extend(first_page_items)

    clean_kw = item_keyword.replace(" ", "").strip()
    parsed = []
    for it in all_raw_items:
        it_nm = str(it.get("item_nm", ""))
        vrty_nm = str(it.get("vrty_nm", ""))
        clean_it_nm = it_nm.replace(" ", "")
        clean_vrty_nm = vrty_nm.replace(" ", "")

        if clean_kw not in clean_it_nm and clean_kw not in clean_vrty_nm:
            if not matched_codes or str(it.get("item_cd", "")) not in matched_codes:
                continue

        raw_price = str(it.get("pmm_avgprc", "0")).replace(",", "")
        unit_name = str(it.get("unit", "kg"))
        unit_sz = str(it.get("unit_sz", "1")).replace(",", "")

        try:
            price_val = float(raw_price)
        except ValueError:
            price_val = 0.0

        # ★ 단위 환산 핵심 : 1kg당 가격 계산
        kg_factor = convert_to_kg_factor(unit_name, unit_sz)
        price_per_kg = price_val / kg_factor if kg_factor > 0 else price_val

        parsed.append({
            "exmn_ym": str(it.get("exmn_ym", "")),
            "se_nm": str(it.get("se_nm", "미구분")),
            "item_nm": it_nm,
            "vrty_nm": vrty_nm if vrty_nm else "일반",
            "grd_nm": str(it.get("grd_nm", "보통")),
            "unit": unit_name,
            "unit_sz": unit_sz,
            "orig_price": price_val,
            "price_per_kg": price_per_kg  # 1kg 환산 단가
        })

    df_res = pd.DataFrame(parsed)
    API_CACHE[cache_key] = df_res
    return df_res

def update_dropdown(df_latest_source):
    if df_latest_source.empty:
        dd_item_select.options = []
        return

    options = []
    for idx, row in df_latest_source.iterrows():
        p_kg = int(row['price_per_kg'])
        label = (f"[{row['exmn_ym']}] [{row['se_nm']}] {row['item_nm']} - "
                 f"{row['vrty_nm']} ({row['grd_nm']}) | 조사단위:{row['unit_sz']}{row['unit']} ➔ 1kg 환산단가: ({p_kg:,}원/kg)")
        options.append((label, idx))
    dd_item_select.options = options

def on_search_clicked(b):
    global df_searched_raw, df_searched_latest
    keyword = txt_search.value.strip()
    se_code = dd_se_cd.value
    s_date = txt_start_date.value.strip()
    e_date = txt_end_date.value.strip()

    if not keyword:
        lbl_status.value = "<b style='color:red;'>품목명을 입력하세요.</b>"
        return

    lbl_status.value = f"<b>상태:</b> '{keyword}' 수집 및 영양 DB 정밀 매칭 중..."

    # 1. 정밀 수분 함량 추산
    moisture_val = get_moisture_content_accurate(keyword)
    num_moisture.value = moisture_val

    with out_table:
        clear_output(wait=True)
    with out_chart:
        clear_output(wait=True)

    # 2. 가격 데이터 수집 및 1kg 환산 적용
    df_fetched = fetch_data_smart(keyword, se_code, s_date, e_date)

    if not df_fetched.empty:
        df_searched_raw = df_fetched.copy()
        df_sorted = df_searched_raw.sort_values(by="exmn_ym", ascending=False)
        df_searched_latest = df_sorted.drop_duplicates(
            subset=["se_nm", "item_nm", "vrty_nm", "grd_nm"], keep="first"
        ).reset_index(drop=True)

        lbl_status.value = f"<b style='color:green;'>✅ 검색 완료: 최신 {len(df_searched_latest)}건 | 정밀 도출 수분 함량: {moisture_val}% (1kg 환산 적용됨)</b>"
        update_dropdown(df_searched_latest)

        with out_table:
            display(HTML(f"<h4>📊 [{keyword}] 최신 원물 가격 정보 (1kg 환산 단가 기준)</h4>"))
            cols = ['exmn_ym', 'se_nm', 'item_nm', 'vrty_nm', 'grd_nm', 'unit_sz', 'unit', 'orig_price', 'price_per_kg']
            df_display = df_searched_latest[cols].copy()
            df_display.columns = ['조사연월', '구분', '품목명', '품종명', '등급', '조사수량', '조사단위', '조사가격(원)', '1kg환산단가(원/kg)']
            display(df_display)
    else:
        df_searched_raw = pd.DataFrame()
        df_searched_latest = pd.DataFrame()
        lbl_status.value = f"<b style='color:red;'>'{keyword}'에 대한 데이터가 없습니다.</b>"
        update_dropdown(pd.DataFrame())

def draw_monthly_chart(change):
    if dd_item_select.value is None or df_searched_raw.empty:
        return
    idx = dd_item_select.value
    if idx not in df_searched_latest.index:
        return

    selected_row = df_searched_latest.loc[idx]
    cond = (
        (df_searched_raw['se_nm'] == selected_row['se_nm']) &
        (df_searched_raw['item_nm'] == selected_row['item_nm']) &
        (df_searched_raw['vrty_nm'] == selected_row['vrty_nm']) &
        (df_searched_raw['grd_nm'] == selected_row['grd_nm'])
    )
    df_history = df_searched_raw[cond].copy()
    if df_history.empty:
        return

    df_history['year_month_fmt'] = pd.to_datetime(df_history['exmn_ym'], format='%Y%m', errors='coerce').dt.strftime('%Y-%m')
    monthly_avg = df_history.groupby('year_month_fmt')['price_per_kg'].mean().reset_index().sort_values('year_month_fmt')

    with out_chart:
        clear_output(wait=True)
        fig = px.line(
            monthly_avg, x='year_month_fmt', y='price_per_kg', markers=True,
            title=f"<b>📈 {selected_row['item_nm']} ({selected_row['vrty_nm']}) 1kg당 가격 추이</b>",
            labels={'year_month_fmt': '조사연월', 'price_per_kg': '1kg당 단가(원)'}
        )
        fig.update_traces(line_color='#0d47a1', line_width=2.5)
        fig.update_layout(height=350, margin=dict(l=20, r=20, t=40, b=20))
        fig.show()

def on_calc_powder_clicked(b):
    if dd_item_select.value is None or df_searched_latest.empty:
        lbl_powder_result.value = "<b style='color:red;'>선택된 원물이 없습니다.</b>"
        return

    idx = dd_item_select.value
    if idx not in df_searched_latest.index:
        return

    row = df_searched_latest.loc[idx]

    raw_input_kg = num_raw_weight.value # 투입량 (kg)
    moisture_pct = num_moisture.value  # 수분 함량 (%)
    mfg_cost = num_mfg_cost.value      # 제조원가 (원)

    raw_unit_price_per_kg = float(row['price_per_kg']) # 1kg당 원물 단가

    total_raw_cost = raw_unit_price_per_kg * raw_input_kg # 원물 총 비용
    total_batch_cost = total_raw_cost + mfg_cost          # 원물 총 비용 + 제조원가

    solid_pct = max(0.0, 100.0 - moisture_pct)
    powder_yield_kg = raw_input_kg * (solid_pct / 100.0) # 동결건조 분말 수율 (kg)
    powder_cost_per_kg = total_batch_cost / powder_yield_kg if powder_yield_kg > 0 else 0.0

    lbl_powder_result.value = f"""
    <div style='background-color:#e8f5e9; padding:15px; border-radius:8px; border:1px solid #4caf50;'>
        <h4 style='margin-top:0; color:#2e7d32;'>🌱 [{row['item_nm']} - {row['vrty_nm']}] 동결건조 분말 원가 산출 결과</h4>
        <table style='width:100%; border-collapse:collapse; font-size:14px;'>
            <tr style='border-bottom:1px solid #c8e6c9;'>
                <td style='padding:6px;'><b>1kg당 환산 원물 단가:</b></td>
                <td style='padding:6px; text-align:right;'>{raw_unit_price_per_kg:,.1f} 원/kg</td>
            </tr>
            <tr style='border-bottom:1px solid #c8e6c9;'>
                <td style='padding:6px;'><b>원물 투입량 ({raw_input_kg:,.0f} kg) 총 비용:</b></td>
                <td style='padding:6px; text-align:right;'>{total_raw_cost:,.0f} 원</td>
            </tr>
            <tr style='border-bottom:1px solid #c8e6c9;'>
                <td style='padding:6px;'><b>수분 함량 / 고형분 함량:</b></td>
                <td style='padding:6px; text-align:right;'>{moisture_pct:.1f}% / <b>{solid_pct:.1f}%</b></td>
            </tr>
            <tr style='border-bottom:1px solid #c8e6c9;'>
                <td style='padding:6px;'><b>🧊 동결건조 분말 최종 생산량:</b></td>
                <td style='padding:6px; text-align:right; font-weight:bold; color:#1565c0;'>{powder_yield_kg:,.2f} kg</td>
            </tr>
            <tr style='border-bottom:1px solid #c8e6c9;'>
                <td style='padding:6px;'><b>추가 가공 / 제조원가:</b></td>
                <td style='padding:6px; text-align:right;'>+ {mfg_cost:,.0f} 원</td>
            </tr>
            <tr style='background-color:#c8e6c9;'>
                <td style='padding:8px;'><b>총 생산 비용 (원물비 + 제조비):</b></td>
                <td style='padding:8px; text-align:right; font-weight:bold;'>{total_batch_cost:,.0f} 원</td>
            </tr>
        </table>
        <div style='margin-top:12px; padding:10px; background-color:#ffffff; border-radius:5px; text-align:center;'>
            <span style='font-size:18px; color:#c62828;'><b>✨ 동결건조 분말 원료 kg 당 최종 원가: {powder_cost_per_kg:,.0f} 원/kg</b></span>
        </div>
    </div>
    """

# 이벤트 바인딩 및 위젯 출력
btn_search.on_click(on_search_clicked)
btn_calc_powder.on_click(on_calc_powder_clicked)
dd_item_select.observe(draw_monthly_chart, names='value')

display(header_html)
display(widgets.HBox([txt_search, dd_se_cd, txt_start_date, txt_end_date, btn_search]))
display(lbl_status)
display(widgets.HBox([out_table, out_chart]))
display(widgets.HTML("<hr><h3>🧪 원물 선택 & 동결건조 분말 제조원가 계산기</h3>"))
display(dd_item_select)
display(widgets.HBox([num_raw_weight, num_moisture, num_mfg_cost, btn_calc_powder]))
display(lbl_powder_result)

ℹ️ 기본 추정 데이터 세트로 작동합니다.


HTML(value='\n<div style="background-color:#0d47a1; padding:14px; border-radius:8px; color:white;">\n    <h2 s…

HTML(value='<b>상태:</b> 품목명을 입력하고 [원물 가격 조회]를 클릭하세요.')

HTML(value='<hr><h3>🧪 원물 선택 & 동결건조 분말 제조원가 계산기</h3>')

Dropdown(description='원물 선택:', layout=Layout(width='55%'), options=(), value=None)

HTML(value="<div style='padding:10px; border:1px solid #ddd; border-radius:5px;'>원물 선택 후 원가 계산을 진행하세요.</div>")